# What the Notebook Is For
## The notebook is not part of the app. It's for:

* Exploration — trying things before committing code
* 
* Documentation — showing the reasoning behind decisions
* 
* Teaching — walking through the data step by step
* 
* Sharing — the kind of notebook you'd attach to a project report

Think of it as "my working notes" that stays in the repo.

# E-Commerce Sales Analytics — Exploration Notebook

**Purpose:** Explore the dataset, verify assumptions, and document
the reasoning behind the dashboard's decisions.

**Author:** Business Analyst

**Project:** E-Commerce Sales Performance & Customer Analytics

**Date:** 2026

In [26]:
# Standard imports
import sys
import os
from pathlib import Path

# Make sure the project root is on the path
project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

print("Pandas version:", pd.__version__)
print("NumPy version:", np.__version__)
print("Project root:", project_root)

Pandas version: 2.3.3
NumPy version: 2.2.6
Project root: d:\Python\E-Commerce Sales Performance & Customer Analytics\ecommerce_analytics


## 1. Load the Raw Data

We start with the raw CSV exactly as it arrives from the source system.
No cleaning, no transformation — we first look at what we have.

In [27]:
from src.data_loader import load_default_data

df_raw = load_default_data()

print("Shape:", df_raw.shape)
print()
print("Columns:", list(df_raw.columns))
print()
df_raw.head()

Shape: (30, 11)

Columns: ['order_id', 'order_date', 'customer_id', 'product_id', 'product_name', 'category', 'region', 'quantity', 'unit_price', 'discount', 'sales']



,order_id,order_date,customer_id,product_id,product_name,category,region,quantity,unit_price,discount,sales
0,ORD1001,2026-01-05,C1001,P2001,Wireless Mouse,Electronics,North,2,799,10,1438.20
1,ORD1002,2026-01-06,C1002,P2002,Cotton T-Shirt,Clothing,South,3,499,5,1422.15
2,ORD1003,2026-01-07,C1003,P2003,Study Table,Furniture,East,1,4999,15,4249.15
3,ORD1004,2026-01-08,C1004,P2004,Face Wash,Beauty,West,5,199,0,995.00
4,ORD1005,2026-01-09,C1005,P2005,Yoga Mat,Sports,North,2,899,10,1618.20


## 2. Initial Inspection

Check data types, missing values, and basic distribution.
This tells us what cleaning will be required.

In [28]:
df_raw.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 30 entries, 0 to 29
Data columns (total 11 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   order_id      30 non-null     object 
 1   order_date    30 non-null     object 
 2   customer_id   30 non-null     object 
 3   product_id    30 non-null     object 
 4   product_name  30 non-null     object 
 5   category      30 non-null     object 
 6   region        30 non-null     object 
 7   quantity      30 non-null     int64  
 8   unit_price    30 non-null     int64  
 9   discount      30 non-null     int64  
 10  sales         30 non-null     float64
dtypes: float64(1), int64(3), object(7)
memory usage: 2.7+ KB


In [29]:
# Missing values per column
missing = df_raw.isna().sum().sort_values(ascending=False)
missing = missing[missing > 0]

if missing.empty:
    print("No missing values found.")
else:
    print("Missing values per column:")
    print(missing)

No missing values found.


In [30]:
# Duplicate orders
dup_count = df_raw["order_id"].duplicated().sum()
print(f"Duplicate order_id rows: {dup_count}")

Duplicate order_id rows: 0


## 3. Validation

Run the same validation checks the dashboard uses, so we can
see what would be flagged in the Data Quality page.

In [31]:
from src.validation import run_all_checks

report = run_all_checks(df_raw)

print("Valid:", report["valid"])
print("Total rows:", report["total_rows"])
print("Duplicate orders:", report["duplicate_orders"])
print("Invalid quantity rows:", report["invalid_quantity_count"])
print("Invalid price rows:", report["invalid_price_count"])
print("Invalid discount rows:", report["invalid_discount_count"])
print("Invalid date rows:", report["invalid_date_count"])

Valid: True
Total rows: 30
Duplicate orders: 0
Invalid quantity rows: 0
Invalid price rows: 0
Invalid discount rows: 0
Invalid date rows: 0


## 4. Clean the Data

Apply the same cleaning pipeline the dashboard uses.
After this step, `df` is what all downstream analysis works on.

In [32]:
from src.data_cleaning import clean_data

df = clean_data(df_raw, verbose=False)

print("Shape after cleaning:", df.shape)
print()
print("Columns after cleaning:", list(df.columns))
print()
df.head()

Shape after cleaning: (30, 17)

Columns after cleaning: ['order_id', 'order_date', 'customer_id', 'product_id', 'product_name', 'category', 'region', 'quantity', 'unit_price', 'discount', 'sales', 'gross_sales', 'discount_amount', 'net_sales', 'order_year', 'order_month', 'order_month_name']



,order_id,order_date,customer_id,product_id,product_name,category,region,quantity,unit_price,discount,sales,gross_sales,discount_amount,net_sales,order_year,order_month,order_month_name
0,ORD1001,2026-01-05,C1001,P2001,Wireless Mouse,Electronics,North,2,799,10,1438.20,1598,159.80,1438.20,2026,1,Jan
1,ORD1002,2026-01-06,C1002,P2002,Cotton T-Shirt,Clothing,South,3,499,5,1422.15,1497,74.85,1422.15,2026,1,Jan
2,ORD1003,2026-01-07,C1003,P2003,Study Table,Furniture,East,1,4999,15,4249.15,4999,749.85,4249.15,2026,1,Jan
3,ORD1004,2026-01-08,C1004,P2004,Face Wash,Beauty,West,5,199,0,995.00,995,0.00,995.00,2026,1,Jan
4,ORD1005,2026-01-09,C1005,P2005,Yoga Mat,Sports,North,2,899,10,1618.20,1798,179.80,1618.20,2026,1,Jan


## 5. KPI Snapshot

Calculate the same KPIs that appear at the top of the dashboard.

In [33]:
from src.kpi import get_all_kpis, format_number

kpis = get_all_kpis(df)

print("Total Sales       :", format_number(kpis["total_sales"]))
print("Total Orders      :", f"{kpis['total_orders']:,}")
print("Total Customers   :", f"{kpis['total_customers']:,}")
print("Total Units       :", f"{kpis['total_units']:,}")
print("Avg Order Value   :", format_number(kpis["average_order_value"]))
print("Median Order Value:", format_number(kpis["median_order_value"]))

Total Sales       : ₹1.08 L
Total Orders      : 30
Total Customers   : 20
Total Units       : 90
Avg Order Value   : ₹3,609
Median Order Value: ₹2,628


## 6. Descriptive Statistics

The core statistical summary of the `sales` column.
This is what powers the Statistics page.

In [34]:
from src.statistics import full_statistics

stats = full_statistics(df["sales"])

for k, v in stats.items():
    print(f"{k:15s}: {v:,}")

mean           : 3,609.08
median         : 2,628.38
mode           : 897.0
variance       : 7,908,151.0
std_dev        : 2,812.14
min            : 897.0
max            : 14,397.6
Q1             : 1,892.64
Q2             : 2,628.38
Q3             : 4,947.23
IQR            : 3,054.59
outlier_count  : 1


## 7. Visualization — Sales Distribution

A histogram shows the shape of the data.
If it's skewed, mean and median will differ.

In [35]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 4))
plt.hist(df["sales"], bins=20, edgecolor="black")
plt.axvline(df["sales"].mean(), color="red", linestyle="--", label="Mean")
plt.axvline(df["sales"].median(), color="green", linestyle="--", label="Median")
plt.title("Distribution of Sales")
plt.xlabel("Sales (₹)")
plt.ylabel("Frequency")
plt.legend()
plt.tight_layout()


## 8. Category & Region Snapshot

Same groupings the dashboard uses.

In [36]:
from src.sales_analysis import sales_by_category, sales_by_region

print("SALES BY CATEGORY")
print(sales_by_category(df).to_string(index=False))
print()
print("SALES BY REGION")
print(sales_by_region(df).to_string(index=False))

SALES BY CATEGORY
   category    sales
  Furniture 36143.25
Electronics 32035.95
   Clothing 16375.65
     Sports 14173.60
     Beauty  9544.00

SALES BY REGION
region    sales
  West 32740.10
  East 29810.40
 South 23350.65
 North 22371.30


## 9. Outlier Investigation

Using the IQR method. Remember: outliers are **flagged**, not removed.
A high-value order may be a legitimate bulk purchase.

In [37]:
from src.statistics import (
    get_outlier_bounds,
    get_outlier_rows,
    count_outliers,
)

lower, upper = get_outlier_bounds(df["sales"])
count = count_outliers(df["sales"])

print(f"Lower bound: {lower:,}")
print(f"Upper bound: {upper:,}")
print(f"Outliers detected: {count}")
print()

if count > 0:
    outliers = get_outlier_rows(df, "sales")
    print("Outlier rows:")
    print(outliers[[
        "order_id", "order_date", "customer_id",
        "category", "region", "sales"
    ]].to_string(index=False))

Lower bound: -2,689.24
Upper bound: 9,529.11
Outliers detected: 1

Outlier rows:
order_id order_date customer_id    category region   sales
 ORD1022 2026-01-26       C1020 Electronics   West 14397.6


## 10. Correlation Analysis

Check whether discount, quantity, and unit_price relate to sales.
Correlation does not imply causation.

In [38]:
from src.statistics import get_correlation, get_covariance, interpret_correlation

pairs = [
    ("discount", "sales"),
    ("quantity", "sales"),
    ("unit_price", "sales"),
]

for x, y in pairs:
    r = get_correlation(df, x, y)
    cov = get_covariance(df, x, y)
    print(f"{x:12s} vs {y:8s} → r = {r:+.3f}  ({interpret_correlation(r)})  cov = {cov:,.2f}")

discount     vs sales    → r = +0.543  (Moderate relationship.)  cov = 11,598.05
quantity     vs sales    → r = -0.222  (Weak relationship.)  cov = -1,364.25
unit_price   vs sales    → r = +0.738  (Strong relationship.)  cov = 4,916,548.55


## 11. Insight Generation

Plain-English sentences from the data.
These are the same insights shown in the "Business Insights" section.

In [39]:
from src.insights import generate_insights, generate_recommendations

print("BUSINESS INSIGHTS")
print("=" * 60)
for i, line in enumerate(generate_insights(df), 1):
    print(f"{i}. {line}")
    print()

print("RECOMMENDATIONS")
print("=" * 60)
for i, line in enumerate(generate_recommendations(df), 1):
    print(f"{i}. {line}")
    print()

BUSINESS INSIGHTS
1. The business recorded ₹1.08 L in total sales across 30 orders from 20 unique customers.

2. **Furniture** is the top category with ₹36,143 in sales, contributing 33.4% of total revenue.

3. **West** region leads with ₹32,740 in sales.

4. **North** region shows the lowest sales at ₹22,371. Consider reviewing demand and distribution there.

5. Top product by revenue is **Smart Watch** (₹19,497).

6. Average order value (₹3,609) is much higher than the median (₹2,628). A few very large orders are pulling the average up.

7. **1 transaction(s)** (3.3%) fall outside the normal order-value range and may require investigation.

8. Discount and sales show a correlation of **0.5428** (moderate relationship.) This does not prove that discount causes sales.

RECOMMENDATIONS
1. Review inventory levels for **Furniture** to ensure top-selling products stay in stock.

2. Investigate the low sales in **North** region — could be pricing, distribution, or local demand.

3. Review t

## 12. Measure of Dispersion — Classroom Examples

Tie the classroom concepts (Range, Variance, Standard Deviation,
Empirical Rule) to the same tools used on real data.

In [40]:
# Example: temperature range
temps = pd.Series([18, 22, 25, 23, 19, 20, 21])
print("Temperature Range:", temps.max() - temps.min(), "°C")

# Example: Store A vs Store B variance
store_a = pd.Series([5, 5, 5, 5, 5])
store_b = pd.Series([1, 3, 5, 7, 9])
print("Store A variance:", store_a.var())
print("Store B variance:", store_b.var())

# Example: Student X vs Y standard deviation
x = pd.Series([70, 75, 80, 85, 90])
y = pd.Series([40, 60, 80, 100, 120])
print("Student X std:", round(x.std(), 2))
print("Student Y std:", round(y.std(), 2))

Temperature Range: 7 °C
Store A variance: 0.0
Store B variance: 10.0
Student X std: 7.91
Student Y std: 31.62


In [41]:
# Empirical Rule on IQ scores (mean=100, sd=15)
mean_iq, sd_iq = 100, 15

for k in (1, 2, 3):
    low = mean_iq - k * sd_iq
    high = mean_iq + k * sd_iq
    pct = {1: "68%", 2: "95%", 3: "99.7%"}[k]
    print(f"±{k} SD ({pct}): {low} to {high}")

±1 SD (68%): 85 to 115
±2 SD (95%): 70 to 130
±3 SD (99.7%): 55 to 145


## 13. Conclusion

What we verified:

- The raw data loads cleanly with all required columns present
- Validation catches invalid rows, duplicates, and bad dates
- Cleaning adds derived columns (`gross_sales`, `discount_amount`, `net_sales`)
- KPIs are computed dynamically from the data
- Statistics (mean, median, IQR, etc.) match the dashboard output
- Outliers are flagged, not removed
- Correlation shows relationships but never claims causation
- Insights are generated in plain English

**Everything the dashboard shows is reproducible in this notebook.**